# 05 — LTV Segmentation and Business Simulation

Converting LTV and churn predictions into marketing decisions.

**Project:** Marketing Analytics Causal & LTV Lab  
**Phase:** Phase 1 — Customer Analytics, Retention, Churn and LTV Baseline

> This notebook is designed as a hands-on learning notebook. Run each section, inspect the output, and discuss the interpretation before moving to the next step.


## 1. Notebook objective

This notebook turns predictions into business decisions.

We will create:

- LTV segments
- Profitability simulation
- CAC sensitivity analysis
- Retention targeting strategy
- Support prioritization strategy

This is the notebook that connects ML output to marketing actions.


In [ ]:
# Core imports
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix
)
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_RAW = Path("../data/raw/digital_wallet_ltv_dataset.csv")
DATA_PROCESSED = Path("../data/processed")
REPORTS = Path("../reports")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PROCESSED / 'wallet_ltv_predictions.csv') if (DATA_PROCESSED / 'wallet_ltv_predictions.csv').exists() else pd.read_csv(DATA_RAW)
df.head()


## 2. Ensure required scores exist


In [ ]:
if "predicted_ltv" not in df.columns:
    print("predicted_ltv not found. Using actual LTV as placeholder. Run 04_ltv_prediction.ipynb first.")
    df["predicted_ltv"] = df["LTV"]

if "churn_risk_score" not in df.columns:
    print("churn_risk_score not found. Creating recency-based proxy. Run 03_churn_analysis.ipynb first.")
    df["churn_risk_score"] = (df["Last_Transaction_Days_Ago"] - df["Last_Transaction_Days_Ago"].min()) / (
        df["Last_Transaction_Days_Ago"].max() - df["Last_Transaction_Days_Ago"].min()
    )


## 3. LTV segmentation


In [ ]:
df["ltv_segment"] = pd.qcut(
    df["predicted_ltv"].rank(method="first"),
    q=4,
    labels=["Low", "Medium", "High", "Very High"]
)

segment_summary = (
    df.groupby("ltv_segment")
    .agg(
        customers=("Customer_ID", "count"),
        avg_predicted_ltv=("predicted_ltv", "mean"),
        avg_actual_ltv=("LTV", "mean"),
        avg_churn_risk=("churn_risk_score", "mean"),
        avg_satisfaction=("Customer_Satisfaction_Score", "mean"),
        avg_transactions=("Total_Transactions", "mean")
    )
)

display(segment_summary)


## 4. CAC profitability simulation


In [ ]:
cac_values = [10, 20, 30, 40, 50, 75, 100]
rows = []

for cac in cac_values:
    profitable_rate = (df["predicted_ltv"] > cac).mean()
    avg_profit = (df["predicted_ltv"] - cac).mean()
    rows.append({
        "cac": cac,
        "profitable_customer_rate": profitable_rate,
        "avg_expected_profit": avg_profit
    })

cac_simulation = pd.DataFrame(rows)
display(cac_simulation)

plt.figure(figsize=(7, 4))
plt.plot(cac_simulation["cac"], cac_simulation["avg_expected_profit"], marker="o")
plt.axhline(0, linestyle="--")
plt.title("CAC Sensitivity: Average Expected Profit")
plt.xlabel("CAC")
plt.ylabel("Predicted LTV - CAC")
plt.show()


## 5. Retention campaign targeting simulation


In [ ]:
# Example assumptions
incentive_cost = 5.0
expected_retention_lift = 0.08

df["target_score"] = df["predicted_ltv"] * df["churn_risk_score"]

df["target_decile"] = pd.qcut(
    df["target_score"].rank(method="first"),
    q=10,
    labels=False
) + 1

# Expected incremental value approximation:
# if retained, expected value is proportional to predicted LTV
df["expected_incremental_value"] = df["predicted_ltv"] * expected_retention_lift
df["expected_campaign_profit"] = df["expected_incremental_value"] - incentive_cost

targeted = df[df["target_decile"] == 10]
blanket = df.copy()

simulation = pd.DataFrame([
    {
        "strategy": "blanket_targeting",
        "customers_targeted": len(blanket),
        "avg_expected_profit": blanket["expected_campaign_profit"].mean(),
        "total_expected_profit": blanket["expected_campaign_profit"].sum()
    },
    {
        "strategy": "top_decile_targeting",
        "customers_targeted": len(targeted),
        "avg_expected_profit": targeted["expected_campaign_profit"].mean(),
        "total_expected_profit": targeted["expected_campaign_profit"].sum()
    }
])

display(simulation)


## 6. Segment-level targeting policy


In [ ]:
policy = (
    df.groupby(["ltv_segment", "target_decile"])
    .agg(
        customers=("Customer_ID", "count"),
        avg_predicted_ltv=("predicted_ltv", "mean"),
        avg_churn_risk=("churn_risk_score", "mean"),
        avg_expected_campaign_profit=("expected_campaign_profit", "mean")
    )
    .reset_index()
    .sort_values("avg_expected_campaign_profit", ascending=False)
)

display(policy.head(20))


## 7. Write business recommendations


In [ ]:
top_segments = (
    df.groupby("ltv_segment")
    .agg(
        customers=("Customer_ID", "count"),
        avg_predicted_ltv=("predicted_ltv", "mean"),
        avg_churn_risk=("churn_risk_score", "mean"),
        avg_campaign_profit=("expected_campaign_profit", "mean")
    )
    .sort_values("avg_campaign_profit", ascending=False)
)

recommendation = f'''
# Phase 1 Business Recommendations

## Objective
Translate customer-level LTV and churn-risk predictions into marketing decisions.

## Recommended targeting logic
Prioritize customers with both:
- High predicted LTV
- High churn risk

Avoid blanket incentives when incentive cost exceeds expected incremental value.

## Simulation assumptions
- Incentive cost: {incentive_cost}
- Expected retention lift: {expected_retention_lift}

## Strategy comparison
{simulation.to_markdown(index=False)}

## Segment summary
{top_segments.to_markdown()}

## Limitations
This simulation is not causal. The assumed retention lift is hypothetical.
In later phases, uplift modeling and A/B testing will estimate incremental impact more directly.
'''

(REPORTS / "phase_1_business_recommendations.md").write_text(recommendation)
print("Saved report: reports/phase_1_business_recommendations.md")


## Discussion prompts

1. Why is high churn risk alone not enough for targeting?
2. Why is high LTV alone not enough?
3. Why is this simulation not causal?
4. What would we need to estimate true incremental retention lift?
